<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will compare three supervised classification methods: Logistic Regression, Decision Tree, and Random Forest.

The target, `is_declining_proxy`, is binary, while the final goal is to rank content pages by their risk of a meaningful decline in search impressions. Each model can produce a score or probability that can be used to rank pages.

I chose to compare all three methods because they provide different levels of complexity. Logistic Regression gives a simple and interpretable reference model, a Decision Tree can capture non-linear decision rules while remaining readable, and Random Forest can capture more complex relationships between the available features.

I will compare their performance against the Week 4 rule-based baseline using the same evaluation data and ranking metrics. The final comparison will show whether the additional model complexity provides a measurable improvement over the simpler approaches.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped client holdout for the train/test split.

Pages from the same `client_hash_id` will stay together, so a client cannot appear in both the training and test sets. This allows the evaluation to measure how well the models generalize to clients that were not seen during training.

The experiment is also time-aware. The model features come only from March 1–15, 2026, while `is_declining_proxy` is calculated from the later March 16–31 outcome window. This keeps future outcome information out of the model features.

Together, the time-aware feature/outcome design and the grouped client split provide an honest evaluation for this task.


## 3. Train + compare vs my baseline

Three models were compared: Logistic Regression, Decision Tree, and Random Forest. Hyperparameters were selected using GroupKFold cross-validation on the training clients only. The final test clients were not used for model selection.

Random Forest had the best cross-validation Precision@100 at **66.4%**, compared with **63.8%** for Logistic Regression and **61.2%** for Decision Tree. I therefore selected Random Forest before evaluating the models on the final held-out test set.

For a fair comparison, the Week 4 baseline rule was re-evaluated on the same frozen Week 5 test clients. This does not replace the original Week 4 baseline result of **39% Precision@100**. On the Week 5 held-out clients, the baseline achieved **46% Precision@100**.

| Model                                                  | Precision@10 | Precision@100 | Average Precision | ROC-AUC |
| ------------------------------------------------------ | -----------: | ------------: | ----------------: | ------: |
| Logistic Regression                                    |          70% |           79% |             0.547 |   0.666 |
| Random Forest                                          |          70% |           78% |             0.574 |   0.689 |
| Decision Tree                                          |          60% |           77% |             0.555 |   0.678 |
| Week 4 baseline, re-evaluated on the same test clients |          50% |           46% |             0.411 |   0.519 |

Although Logistic Regression reached **79% Precision@100** on this particular test set, I did not switch models after seeing the test results. Random Forest remains the selected model because it had the strongest grouped cross-validation result before the final test was opened.

The selected Random Forest achieved **78% Precision@100**, compared with **46%** for the baseline on the same held-out clients. This is an absolute improvement of **32 percentage points**.

The momentum features also added useful information. In the Logistic Regression ablation check, using level features alone produced **42.8% CV Precision@100**, while level plus momentum features reached **63.8%**. This suggests that recent direction of change provides useful signal beyond current traffic level alone.

The result should still be treated as directional. The final test set contains only **7 clients**, and their decline rate was **38.9%**, compared with **28.2%** in the training clients. A different client sample or time period could produce a different score.

---

## 4. Errors and interpretation

The selected model performed well at ranking likely declines, but it was not correct in every case. I inspected high-confidence false positives and low-ranked true declines to understand where the errors came from.

### False positives

The three strongest false positives received risk scores of approximately **0.94–0.95**. Before the outcome window, their impression momentum had fallen sharply by approximately **91%, 85%, and 83%**.

These pages looked risky because they showed a strong negative movement before the future window. However, they did not satisfy the defined decline target during the outcome period.

This shows an important limitation: a sharp short-term drop is a useful warning signal, but it does not always continue. Some pages may recover or stabilize after the feature window.

### Missed declines

The lowest-ranked true declines had risk scores of approximately **0.10**.

Two of these pages had very low search volume and positive impression momentum before the outcome window. Their impression momentum was approximately **+64.5%** and **+9.4%**, so there was little evidence of an approaching decline at prediction time.

Another missed decline had much higher traffic but only a small negative impression movement of about **-5.9%**. The later decline therefore occurred without a strong pre-outcome momentum signal.

These cases suggest that the model can miss declines when the warning signs are weak, when traffic volume is very low, or when a page appears stable or improving immediately before the outcome window.

### What the model relies on

Permutation importance showed that the three strongest features were:

1. **Impression momentum (`imp_momentum_log`)**
2. **First-half impression volume (`log_imp_first_half`)**
3. **First-half CTR (`ctr_first_half`)**

The strongest feature was impression momentum, which is consistent with the purpose of the model: identifying pages whose search visibility is beginning to weaken before the future outcome period.

All of these features are calculated from the pre-outcome window. No March 16–31 outcome information, client ID, or content ID is used as a model feature.

### Interpretation

The model should be used as a **decision-support ranking tool**, not as proof that a page will decline or as a prediction of Google's algorithm.

Its practical use is to move pages with stronger observed warning signals toward the top of a review queue. The error analysis shows why human review is still needed: strong negative momentum can recover, while some future declines occur without a clear warning signal.

Overall, the held-out test provides evidence that the selected model ranks declining pages more effectively than the baseline rule on these clients, while the small number of test clients means the exact performance estimate should be validated on additional clients or future time periods.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.